# Frozen Encoder Geometry Refiner

Train the geometry transformer and optional SE3 refiner from a frozen FoldTree2 encoder. The notebook uses `learn_geometry_lightning.py` so the training path stays aligned with the script, then visualizes TensorBoard losses and one example structure with true, coarse, and SE3-refined CA/C/CB/N atoms.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy
import os
import subprocess
import sys
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

start = Path.cwd()
candidates = [start, *start.parents]
REPO = next((p for p in candidates if (p / "learn_geometry_lightning.py").exists()), None)
if REPO is None:
    raise RuntimeError(f"Could not find learn_geometry_lightning.py above {start}")

os.chdir(REPO)
os.environ.setdefault("PYTHONPATH", str(REPO.parent))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

print("repo:", REPO)
print("python:", sys.executable)
print("cuda:", torch.cuda.is_available())

## Configuration

The defaults below are intentionally small enough for a smoke run. Increase `epochs`, remove `limit_train_batches`, and widen the decoder once the losses are finite.

In [ ]:
cfg = SimpleNamespace(
    dataset="notebooks/structs_training_mk2.h5",
    pretrained_encoder_path="/home/dmoi/projects/foldtree2/models/production/30char_minimal_decoder/final_30char_contacts_aa_encoder_full_epoch_52.pt",
    pretrained_encoder_full_path="/home/dmoi/projects/foldtree2/models/production/30char_minimal_decoder/final_30char_contacts_aa_encoder_full_epoch_52.pt",
    checkpoint_dir="/tmp/foldtree2_geometry_refiner_notebook",
    epochs=5,
    limit_train_batches=12,
    batch_size=1,
    target_effective_batch_size=2,
    learning_rate=5e-4,
    accelerator="cpu",
    devices="1",
    precision="32-true",
    use_se3=True,
    use_se3_atom_refine=True,
    se3_hidden=16,
    se3_depth=1,
    se3_heads=1,
    se3_dim_head=8,
    transformer_width=64,
    transformer_layers=2,
    transformer_nheads=4,
    coarse_backbone_angle_weight=0.05,
    se3_atom_weight=0.05,
    se3_atom_fape_weight=0.25,
    log_every_n_steps=1,
)

Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
cfg

## Train

This trains only the geometry transformer and SE3 decoder. The encoder is loaded from the checkpoint and frozen by `GeometryFocusedModule`.

In [ ]:
cmd = [
    sys.executable,
    "learn_geometry_lightning.py",
    "--dataset", cfg.dataset,
    "--epochs", str(cfg.epochs),
    "--limit-train-batches", str(cfg.limit_train_batches),
    "--batch-size", str(cfg.batch_size),
    "--target-effective-batch-size", str(cfg.target_effective_batch_size),
    "--accelerator", cfg.accelerator,
    "--devices", cfg.devices,
    "--precision", cfg.precision,
    "--pretrained-encoder-path", cfg.pretrained_encoder_path,
    "--pretrained-encoder-full-path", cfg.pretrained_encoder_full_path,
    "--no-use-uncertainty-weighting",
    "--use-coarse-ca-loss",
    "--use-coarse-backbone-loss",
    "--coarse-ca-weight", "1.0",
    "--coarse-ca-step-frame", "prev",
    "--rotation-target-frame", "local",
    "--coarse-backbone-angle-weight", str(cfg.coarse_backbone_angle_weight),
    "--transformer-width", str(cfg.transformer_width),
    "--transformer-layers", str(cfg.transformer_layers),
    "--transformer-nheads", str(cfg.transformer_nheads),
    "--rt-hidden", "64,32,16",
    "--rotation-hidden", "64,32,16",
    "--ca-step-hidden", "64,32,16",
    "--angles-hidden", "64,32,16",
    "--ss-hidden", "32,16,8",
    "--learning-rate", str(cfg.learning_rate),
    "--checkpoint-dir", cfg.checkpoint_dir,
    "--save-top-k", "1",
    "--log-every-n-steps", str(cfg.log_every_n_steps),
]

if cfg.use_se3:
    cmd += [
        "--use-se3",
        "--se3-input-source", "coarse_ca",
        "--se3-hidden", str(cfg.se3_hidden),
        "--se3-depth", str(cfg.se3_depth),
        "--se3-heads", str(cfg.se3_heads),
        "--se3-dim-head", str(cfg.se3_dim_head),
    ]
    if cfg.use_se3_atom_refine:
        cmd += [
            "--use-se3-atom-refine",
            "--se3-atom-weight", str(cfg.se3_atom_weight),
            "--se3-atom-fape-weight", str(cfg.se3_atom_fape_weight),
        ]
else:
    cmd += ["--no-use-se3"]

print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=REPO, env=os.environ.copy())

## Loss Curves

In [ ]:
def latest_lightning_log(root="lightning_logs"):
    root = Path(root)
    versions = sorted(root.glob("version_*"), key=lambda p: p.stat().st_mtime)
    if not versions:
        raise FileNotFoundError(f"No Lightning logs found in {root}")
    return versions[-1]

def load_scalars(logdir):
    ea = EventAccumulator(str(logdir))
    ea.Reload()
    out = {}
    for tag in ea.Tags().get("scalars", []):
        out[tag] = np.array([x.value for x in ea.Scalars(tag)], dtype=np.float32)
    return out

logdir = latest_lightning_log()
scalars = load_scalars(logdir)
print("logdir:", logdir)
print("scalar tags:", sorted(k for k in scalars if "raw" in k or "loss_epoch" in k))

plot_tags = [
    "train/loss_epoch",
    "train/raw_coarse_backbone_atoms",
    "train/raw_coarse_backbone_fape",
    "train/raw_coarse_backbone_angles",
    "train/raw_fape_quat_se3",
    "train/raw_se3_atom_refine",
    "train/raw_se3_atom_fape",
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()
for ax, tag in zip(axes, plot_tags):
    y = scalars.get(tag)
    if y is None or len(y) == 0:
        ax.set_title(tag + "\nmissing")
        ax.axis("off")
        continue
    ax.plot(np.arange(len(y)), y, marker="o", linewidth=1.5)
    ax.set_title(tag.replace("train/raw_", "").replace("train/", ""))
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.25)
for ax in axes[len(plot_tags):]:
    ax.axis("off")
plt.tight_layout()

## Reload Checkpoint And Predict One Example

This cell reconstructs the frozen encoder and decoder modules, loads the Lightning checkpoint, runs one batch through `_compute_total_loss`, and reads the prediction tensors that the loss path stores on the batch.

In [ ]:
from learn_geometry_lightning import (
    GeometryFocusedModule,
    build_decoders,
    build_encoder,
    ensure_edge_attrs_inplace,
    ensure_float32_inplace,
    parse_devices,
)
from foldtree2.src import pdbgraphmk2

def notebook_args():
    return SimpleNamespace(
        pretrained_encoder_path=cfg.pretrained_encoder_path,
        pretrained_encoder_full_path=cfg.pretrained_encoder_full_path,
        fallback_latent_dim=64,
        transformer_width=cfg.transformer_width,
        transformer_layers=cfg.transformer_layers,
        transformer_nheads=cfg.transformer_nheads,
        transformer_dropout=0.05,
        rt_hidden="64,32,16",
        rotation_hidden="64,32,16",
        ca_step_hidden="64,32,16",
        ss_hidden="32,16,8",
        angles_hidden="64,32,16",
        use_se3=cfg.use_se3,
        se3_hidden=cfg.se3_hidden,
        se3_out_channels=96,
        se3_depth=cfg.se3_depth,
        se3_heads=cfg.se3_heads,
        se3_dim_head=cfg.se3_dim_head,
    )

dataset = pdbgraphmk2.StructureDataset(cfg.dataset)
loader = DataLoader(dataset, batch_size=1, shuffle=False)
batch = next(iter(loader))
device = torch.device("cuda" if torch.cuda.is_available() and cfg.accelerator != "cpu" else "cpu")
batch = ensure_edge_attrs_inplace(ensure_float32_inplace(batch.to(device)), edge_dim=1)

args = notebook_args()
encoder, latent_dim = build_encoder(args, copy.deepcopy(batch), device)
se3_num_atom_types = max(4, int(getattr(encoder, "num_embeddings", 20)))
transformer_decoder, se3_decoder = build_decoders(latent_dim, copy.deepcopy(batch), device, cfg.use_se3, args, se3_num_atom_types)

ckpts = sorted(Path(cfg.checkpoint_dir).glob("*.ckpt"), key=lambda p: p.stat().st_mtime)
if not ckpts:
    raise FileNotFoundError(f"No checkpoints found in {cfg.checkpoint_dir}")
ckpt = ckpts[-1]
print("checkpoint:", ckpt)

module = GeometryFocusedModule.load_from_checkpoint(
    str(ckpt),
    encoder=encoder,
    transformer_geom_decoder=transformer_decoder,
    se3_decoder=se3_decoder,
    map_location=device,
)
module.eval().to(device)

with torch.no_grad():
    pred_batch = copy.deepcopy(batch)
    total, raw_terms, weighted_terms, se3_skip, _ = module._compute_total_loss(pred_batch, debug_label="notebook_predict")

print("total:", float(total.detach().cpu()))
print({k: float(v.detach().cpu()) for k, v in raw_terms.items()})
print("identifier:", getattr(pred_batch, "identifier", None))

## Structure Visualizations

Plots are centered independently for visual comparison. Use them to catch gross frame flips, trace collapse, exploding SE3 coordinates, or CB/N/C offsets pointing the wrong way.

In [ ]:
ATOM_COLORS = {"ca": "black", "c": "tab:blue", "cb": "tab:green", "n": "tab:red"}

def node(name):
    if name in pred_batch.node_types and hasattr(pred_batch[name], "x"):
        return pred_batch[name].x.detach().cpu()
    return None

def stack_atoms(prefix):
    atoms = []
    names = ["ca", "c", "cb", "n"]
    for name in names:
        key = "coords" if prefix == "true" and name == "ca" else f"{prefix}_{name}_pred"
        if prefix == "true" and name == "c":
            key = "ccoords"
        elif prefix == "true" and name == "cb":
            key = "cbcoords"
        elif prefix == "true" and name == "n":
            key = "ncoords"
        x = node(key)
        if x is None:
            return None, names
        atoms.append(x)
    return torch.stack(atoms, dim=1), names

def center(x):
    return x - x.reshape(-1, 3).mean(dim=0)

def plot_atoms(ax, atoms, names, title, stride=1):
    atoms = center(atoms)[::stride]
    ca = atoms[:, names.index("ca")]
    ax.plot(ca[:, 0], ca[:, 1], ca[:, 2], color="0.35", linewidth=1.0, alpha=0.8)
    for i, name in enumerate(names):
        pts = atoms[:, i]
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=12, color=ATOM_COLORS[name], label=name, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend(loc="upper right", fontsize=8)

true_atoms, atom_names = stack_atoms("true")
coarse_atoms, _ = stack_atoms("coarse")
se3_atoms, _ = stack_atoms("se3")

fig = plt.figure(figsize=(17, 5))
panels = [(true_atoms, "true"), (coarse_atoms, "coarse initial"), (se3_atoms, "SE3 refined")]
for i, (atoms, title) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, i, projection="3d")
    if atoms is None:
        ax.set_title(title + " missing")
        ax.axis("off")
    else:
        plot_atoms(ax, atoms, atom_names, title)
plt.tight_layout()

In [ ]:
def ca_distance_matrix(atoms):
    ca = atoms[:, atom_names.index("ca")]
    return torch.cdist(ca, ca).numpy()

mats = []
titles = []
for atoms, title in panels:
    if atoms is not None:
        mats.append(ca_distance_matrix(atoms))
        titles.append(title)

fig, axes = plt.subplots(1, len(mats), figsize=(5 * len(mats), 4))
if len(mats) == 1:
    axes = [axes]
for ax, mat, title in zip(axes, mats, titles):
    im = ax.imshow(mat, cmap="magma_r", vmin=0, vmax=min(30, np.nanmax(mat)))
    ax.set_title(title + " CA distances")
    ax.set_xlabel("residue")
    ax.set_ylabel("residue")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()